# Fine-tuning DistilBERT (cahya/distilbert-base-indonesian) untuk Semantic Search
**Pendekatan:** Masked Language Modeling (MLM) — Domain-Adaptive Pretraining

Notebook ini **terpisah** dari notebook utama (`distilbert-cahya.ipynb`) dan merupakan
versi revisi dari pendekatan *contrastive learning*. Alurnya:
1. Load & preprocessing dataset arsip (sama seperti notebook utama)
2. Menyusun teks dokumen sebagai korpus untuk MLM (tanpa perlu pasangan positif/kategori)
3. Fine-tuning DistilBERT dengan objective Masked Language Modeling (menebak token yang di-*mask*)
4. Simpan model hasil fine-tuning (siap dijadikan Kaggle Dataset/Model)
5. Sanity check: bandingkan cosine similarity intra-category vs inter-category sebelum & sesudah fine-tuning

> **Catatan metodologis:** MLM adalah pendekatan *self-supervised* yang tidak menggunakan label
> `categoryId` sama sekali selama pelatihan. Model dilatih untuk memprediksi token yang disembunyikan
> (*masked*) dalam kalimat, sehingga tujuan optimasinya adalah memperkaya pemahaman bahasa pada domain
> dokumen arsip akademik — berbeda dengan *contrastive learning* yang secara eksplisit menata jarak
> antar-dokumen di ruang vektor berdasarkan kategori.


## 1️. Instalasi & Import Library

In [ ]:
# Install library yang dibutuhkan (khusus Kaggle environment)
!pip install -q transformers openpyxl

In [ ]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
from IPython.display import display

# Reproducibility
SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('✅ Semua library berhasil diimport')
print(f'   PyTorch version : {torch.__version__}')
print(f'   Device          : {"GPU" if torch.cuda.is_available() else "CPU"}')


## 2️. Load Dataset

In [ ]:
# ── Sesuaikan path dengan lokasi file di Kaggle (samakan dengan notebook utama) ──
FILE_PATH = '/kaggle/input/datasets/muhammadhaekal74/dataset-arsip/dataset_arsip.xlsx'

df = pd.read_excel(FILE_PATH)

print(f'✅ Dataset berhasil dimuat')
print(f'   Jumlah baris   : {len(df):,}')
print(f'   Jumlah kolom   : {len(df.columns)}')
display(df[['title', 'description', 'keywords', 'categoryId']].head(3))


## 3️. Preprocessing (identik dengan notebook utama)

In [ ]:
def clean_text(text: str) -> str:
    """Bersihkan teks: lowercase, hapus karakter khusus, normalisasi spasi."""
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_combined_text(row: pd.Series) -> str:
    """Gabungkan kolom title + description + keywords menjadi satu teks."""
    parts = [
        clean_text(str(row.get('title', ''))),
        clean_text(str(row.get('description', ''))),
        clean_text(str(row.get('keywords', '')))
    ]
    return ' '.join(p for p in parts if p)


df['combined_text'] = df.apply(build_combined_text, axis=1)
df = df[df['combined_text'].str.strip() != ''].reset_index(drop=True)

print('✅ Preprocessing selesai')
print(f'   Dokumen valid  : {len(df):,}')
print()
print('Distribusi categoryId:')
display(df['categoryId'].value_counts().sort_index())


## 4️. Split Dataset untuk Training MLM

Berbeda dengan pendekatan *contrastive learning* yang membutuhkan pasangan (anchor, positive)
dari `categoryId`, pendekatan MLM **tidak memerlukan label sama sekali** — cukup teks dokumen
itu sendiri sebagai korpus pelatihan. `categoryId` tetap dipertahankan pada split data
(`stratify`) semata-mata agar proporsi kategori pada data train/val tetap representatif,
bukan karena dibutuhkan oleh objective MLM.


In [ ]:
VAL_SIZE = 0.15  # porsi dokumen untuk validasi

train_df, val_df = train_test_split(
    df, test_size=VAL_SIZE, stratify=df['categoryId'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f'✅ Split dokumen: {len(train_df)} train, {len(val_df)} val')
print()
print('Contoh combined_text (train):')
print(train_df['combined_text'].iloc[0])


## 5️. Load Model Dasar (Base Model) untuk MLM

In [ ]:
MODEL_NAME  = 'cahya/distilbert-base-indonesian'
MAX_LENGTH  = 256
BATCH_SIZE  = 16
EPOCHS      = 8
LR          = 2e-5
MLM_PROB    = 0.15   # proporsi token yang di-mask, standar BERT/RoBERTa

print(f'🔄 Memuat model dasar "{MODEL_NAME}" ke {DEVICE.upper()}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# AutoModelForMaskedLM: base encoder DistilBERT + head prediksi token (vocab projector)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)

print('✅ Model dasar (dengan MLM head) berhasil dimuat')


## 6️. Mean Pooling & Fungsi Encoding untuk Sanity Check

Fungsi ini dipakai khusus untuk mengambil representasi *embedding* dokumen (bukan logit prediksi
token), dengan mengambil *hidden state* terakhir dari base encoder — MLM head (`vocab_projector`)
tidak dipakai di sini, karena tujuannya murni untuk mengukur kualitas representasi vektor,
sama seperti pada pendekatan *contrastive learning* sebelumnya.


In [ ]:
def mean_pooling_from_hidden(hidden_states, attention_mask):
    """Mean Pooling dari hidden state terakhir (bukan logit MLM)."""
    mask_expanded  = attention_mask.unsqueeze(-1).float()
    sum_embeddings = (hidden_states * mask_expanded).sum(1)
    sum_mask       = mask_expanded.sum(1).clamp(min=1e-9)
    return sum_embeddings / sum_mask


@torch.no_grad()
def encode_texts_eval(texts, batch_size=BATCH_SIZE):
    """Encode tanpa gradient — dipakai untuk sanity check sebelum/sesudah fine-tuning."""
    model.eval()
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                             max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
        output = model(**encoded, output_hidden_states=True)
        # hidden_states[-1] == last_hidden_state dari base encoder (setara AutoModel biasa)
        last_hidden = output.hidden_states[-1]
        emb = mean_pooling_from_hidden(last_hidden, encoded['attention_mask'])
        emb = F.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings).astype(np.float32)


print('✅ Fungsi mean_pooling_from_hidden & encode_texts_eval siap')


## 6.1 Sanity Check — Sebelum Fine-tuning

In [ ]:
def intra_inter_similarity(df, n_per_cat=15, n_cats=2, seed=SEED):
    rng = random.Random(seed)
    cats = df['categoryId'].value_counts()
    cats = cats[cats >= n_per_cat].index.tolist()
    chosen_cats = rng.sample(cats, min(n_cats, len(cats)))

    sample_df = pd.concat([
        df[df['categoryId'] == c].sample(n=n_per_cat, random_state=seed)
        for c in chosen_cats
    ]).reset_index(drop=True)

    embeddings = encode_texts_eval(sample_df['combined_text'].tolist())
    sim_matrix = embeddings @ embeddings.T
    cats_arr = sample_df['categoryId'].values

    intra_sims, inter_sims = [], []
    n = len(sample_df)
    for i in range(n):
        for j in range(i + 1, n):
            if cats_arr[i] == cats_arr[j]:
                intra_sims.append(sim_matrix[i, j])
            else:
                inter_sims.append(sim_matrix[i, j])

    return np.mean(intra_sims), np.mean(inter_sims), chosen_cats


intra_before, inter_before, chosen_cats = intra_inter_similarity(df)
print('===== SEBELUM FINE-TUNING (MLM) =====')
print(f'   Kategori sampel        : {chosen_cats}')
print(f'   Rata-rata sim intra    : {intra_before:.4f}')
print(f'   Rata-rata sim inter    : {inter_before:.4f}')
print(f'   Selisih (intra-inter)  : {intra_before - inter_before:.4f}')


## 7️. Dataset & DataCollator untuk Training MLM

`DataCollatorForLanguageModeling` dari HuggingFace menangani dua hal otomatis di setiap batch:
1. **Dynamic padding** — menyamakan panjang token dalam satu batch
2. **Random masking** — menyembunyikan `MLM_PROB` (15%) token secara acak, lalu membuat
   `labels` (token asli sebelum di-*mask*, hanya pada posisi yang di-*mask*)

Ini menggantikan fungsi `generate_positive_pairs()` pada pendekatan *contrastive learning*
sebelumnya — MLM tidak memerlukan pembentukan pasangan dokumen sama sekali.


In [ ]:
class TextDataset(Dataset):
    """Dataset sederhana: tokenisasi teks tanpa padding (padding ditangani collator)."""
    def __init__(self, texts, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length)

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=MLM_PROB
)

train_dataset = TextDataset(train_df['combined_text'].tolist(), tokenizer)
val_dataset   = TextDataset(val_df['combined_text'].tolist(), tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=data_collator, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=data_collator, drop_last=True
)

print(f'✅ DataLoader siap — {len(train_loader)} batch train, {len(val_loader)} batch val '
      f'(batch_size={BATCH_SIZE}, mlm_probability={MLM_PROB})')


## 8️. Training — Masked Language Modeling

Untuk setiap batch:
1. `data_collator` men-*mask* ±15% token secara acak dan menyediakan `labels`
2. Model memprediksi token asli pada posisi yang di-*mask* (`vocab_projector` head)
3. Loss = *cross-entropy* antara token prediksi dan token asli, dihitung otomatis oleh
   `AutoModelForMaskedLM` dan dikembalikan lewat `outputs.loss`

Berbeda dengan *symmetric InfoNCE loss* pada pendekatan sebelumnya, di sini tidak ada
konsep anchor/positive, *in-batch negatives*, maupun `TEMPERATURE` — murni prediksi token.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_history = []
val_loss_history = []

for epoch in range(EPOCHS):
    # ── Training ──────────────────────────────────────────────
    model.train()
    epoch_losses = []

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)   # outputs.loss = MLM cross-entropy loss
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    avg_loss = float(np.mean(epoch_losses))
    loss_history.append(avg_loss)

    # ── Validation ────────────────────────────────────────────
    model.eval()
    epoch_val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            epoch_val_losses.append(outputs.loss.item())

    avg_val_loss = float(np.mean(epoch_val_losses))
    val_loss_history.append(avg_val_loss)

    print(f'Epoch {epoch+1:>2}/{EPOCHS} — Train Loss: {avg_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

print()
print('✅ Fine-tuning (MLM) selesai')


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker='o', label='Train Loss')
plt.plot(range(1, EPOCHS + 1), val_loss_history, marker='s', label='Validation Loss')
plt.title('Training vs Validation Loss — MLM Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('MLM Cross-Entropy Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss_curve_mlm.png', dpi=150)
plt.show()


## 9️. Sanity Check — Sesudah Fine-tuning

In [ ]:
intra_after, inter_after, _ = intra_inter_similarity(df, seed=SEED)

comparison = pd.DataFrame({
    'Metrik': ['Rata-rata sim intra-category', 'Rata-rata sim inter-category', 'Selisih (intra - inter)'],
    'Sebelum Fine-tuning': [f'{intra_before:.4f}', f'{inter_before:.4f}', f'{intra_before - inter_before:.4f}'],
    'Sesudah Fine-tuning': [f'{intra_after:.4f}', f'{inter_after:.4f}', f'{intra_after - inter_after:.4f}'],
})
display(comparison)

print()
if (intra_after - inter_after) > (intra_before - inter_before):
    print('✅ Fine-tuning (MLM) meningkatkan separasi antar kategori (indikasi awal positif).')
else:
    print('⚠️ Separasi belum meningkat — wajar untuk MLM karena tidak ada sinyal categoryId')
    print('   selama pelatihan; MLM hanya memperkaya pemahaman bahasa domain arsip secara umum,')
    print('   bukan menata representasi berdasarkan kategori seperti contrastive learning.')


## 9.1 Eksperimen Multi-Seed — Validasi Stabilitas Training

Mengulang **seluruh proses training dari awal** (model dasar fresh) untuk 5 seed berbeda:
`[42, 123, 2024, 7, 99]`. Tujuannya sama seperti pada notebook *contrastive learning*
sebelumnya — membuktikan hasil fine-tuning tidak kebetulan bagus di satu seed saja, dan
menghasilkan **Mean ± Std** dari `final_loss` dan `separation_gap` sebagai bukti
reproducibility metodologis.

> Sel ini menjalankan training 5× penuh (5× lebih lama dari training tunggal). Model dasar
> di-*load* ulang fresh di setiap seed — bukan melanjutkan dari model yang sudah di-fine-tune
> sebelumnya.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# EKSPERIMEN MULTI-SEED — Training Ulang dari Awal untuk 5 Seed (MLM)
# ══════════════════════════════════════════════════════════════════
SEEDS = [42, 123, 2024, 7, 99]

multiseed_results = []

for exp_seed in SEEDS:
    print('=' * 60)
    print(f'🔁 Training MLM dengan SEED = {exp_seed}')
    print('=' * 60)

    torch.manual_seed(exp_seed)
    np.random.seed(exp_seed)
    random.seed(exp_seed)

    # ── Split dengan seed ini ─────────────────────────────────────
    exp_train_df, exp_val_df = train_test_split(
        df, test_size=VAL_SIZE, stratify=df['categoryId'], random_state=exp_seed
    )

    exp_train_dataset = TextDataset(exp_train_df['combined_text'].tolist(), tokenizer)
    exp_val_dataset   = TextDataset(exp_val_df['combined_text'].tolist(), tokenizer)

    exp_train_loader = DataLoader(
        exp_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=data_collator, drop_last=True
    )
    exp_val_loader = DataLoader(
        exp_val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=data_collator, drop_last=True
    )

    # ── Model dasar fresh (reset dari pretrained, bukan lanjutan) ─
    exp_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    exp_optimizer = torch.optim.AdamW(exp_model.parameters(), lr=LR, weight_decay=0.01)

    exp_train_loss_history, exp_val_loss_history = [], []

    for epoch in range(EPOCHS):
        exp_model.train()
        epoch_losses = []
        for batch in exp_train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            exp_optimizer.zero_grad()
            loss = exp_model(**batch).loss
            loss.backward()
            exp_optimizer.step()
            epoch_losses.append(loss.item())
        exp_train_loss_history.append(float(np.mean(epoch_losses)))

        exp_model.eval()
        epoch_val_losses = []
        with torch.no_grad():
            for batch in exp_val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                epoch_val_losses.append(exp_model(**batch).loss.item())
        exp_val_loss_history.append(float(np.mean(epoch_val_losses)))

    final_train_loss = exp_train_loss_history[-1]
    final_val_loss    = exp_val_loss_history[-1]

    # ── Sanity check pakai model hasil seed ini ───────────────────
    global model
    _orig_model = model
    model = exp_model  # sementara dialihkan supaya encode_texts_eval pakai exp_model
    exp_intra, exp_inter, _ = intra_inter_similarity(df, seed=SEED)
    model = _orig_model  # kembalikan

    exp_gap = exp_intra - exp_inter

    multiseed_results.append({
        'seed': exp_seed,
        'final_train_loss': final_train_loss,
        'final_val_loss': final_val_loss,
        'intra_category_sim': exp_intra,
        'inter_category_sim': exp_inter,
        'separation_gap': exp_gap,
    })

    print(f'   Final Train Loss : {final_train_loss:.4f}')
    print(f'   Final Val Loss   : {final_val_loss:.4f}')
    print(f'   Separation Gap   : {exp_gap:.4f}')
    print()

    del exp_model
    torch.cuda.empty_cache()

multiseed_df = pd.DataFrame(multiseed_results)

print('=' * 60)
print(' RINGKASAN EKSPERIMEN MULTI-SEED (TRAINING MLM)')
print('=' * 60)
display(multiseed_df)
print()
print(f"Final Train Loss : {multiseed_df['final_train_loss'].mean():.4f} ± {multiseed_df['final_train_loss'].std():.4f}")
print(f"Final Val Loss    : {multiseed_df['final_val_loss'].mean():.4f} ± {multiseed_df['final_val_loss'].std():.4f}")
print(f"Separation Gap    : {multiseed_df['separation_gap'].mean():.4f} ± {multiseed_df['separation_gap'].std():.4f}")


## 9.2 Langkah Selanjutnya — Training Model Produksi

Setelah melihat `multiseed_df` di atas:

1. Pilih seed terbaik (default rekomendasi: `final_val_loss` terendah — bisa juga pilih yang
   `separation_gap`-nya paling dekat ke rata-rata, untuk representasi yang lebih fair).
2. Ubah `SEED = 7` di **cell konfigurasi awal** (bagian Import Library) menjadi seed terpilih.
3. Jalankan ulang notebook ini **dari cell awal sampai bagian Simpan Model** (bagian 10) —
   proses training tunggal yang sudah ada di notebook ini akan otomatis memakai seed terpilih
   tersebut sebagai model produksi final.

Section eksperimen multi-seed (9.1) di atas **tidak menghasilkan model yang disimpan** —
perannya murni untuk validasi metodologi dan menentukan seed mana yang dipakai untuk
training produksi, sama seperti pada pendekatan *contrastive learning* sebelumnya.


## 10. Simpan Model Hasil Fine-tuning

Hanya **base encoder** (`model.base_model`) yang disimpan — bukan seluruh
`AutoModelForMaskedLM` beserta *head* prediksi token (`vocab_projector`), karena *head*
tersebut hanya dipakai untuk objective MLM saat training dan tidak relevan untuk tahap
pembentukan *embedding* di Notebook 3. Dengan menyimpan base encoder saja, model bisa
langsung di-*load* dengan `AutoModel.from_pretrained(path)` tanpa peringatan
*mismatched/unused weights*, persis seperti pada pendekatan *contrastive learning*
sebelumnya — sehingga kode `encode_texts()` di notebook utama tidak perlu diubah sama sekali.


In [ ]:
SAVE_DIR = './distilbert-arsip-finetuned-mlm'

model.base_model.save_pretrained(SAVE_DIR)   # hanya base encoder, tanpa MLM head
tokenizer.save_pretrained(SAVE_DIR)

print(f'✅ Model & tokenizer tersimpan di: {SAVE_DIR}')
print()


In [ ]:
import shutil
from IPython.display import FileLink

# Zip folder model
shutil.make_archive('distilbert-arsip-finetuned-mlm', 'zip', SAVE_DIR)

# Buat link download
FileLink('distilbert-arsip-finetuned-mlm.zip')
